# Phase 3: Activation-Guided Frequency Masking

Uses model activations to identify important frequencies and mask them intelligently.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print("Setup complete!")

## 2. Configuration

In [ ]:
from datamodules.components.frequency_masker import ActivationGuidedFrequencyMasker
from utils.frequency_importance_tracker import FrequencyImportanceTracker

device = 'cpu'
tracker = FrequencyImportanceTracker(n_mels=128, ema_alpha=0.9, device=device)

masker = ActivationGuidedFrequencyMasker(
    n_bands=2, mask_prob=1.0, max_band_width=16, n_mels=128,
    energy_weight=0.2, importance_weight=0.7, ema_alpha=0.1,
    importance_tracker=tracker, afri_weight=0.0
)

print(f"Config: {masker}")

## 3. Mock Model Setup

In [ ]:
class MockResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

model = MockResNet().to(device).eval()
tracker.register_hook(model)

# Populate tracker
print("Populating tracker...")
for i in range(20):
    spec = torch.randn(1, 1, 100, 128).to(device)
    spec[:, :, :, 80:] *= 2.0  # High frequencies important
    with torch.no_grad():
        _ = model(spec)
print(f"Done! Stats: {tracker.get_stats()}")

## 4. Test Data

In [ ]:
def create_spec(pattern='mixed', shape=(100, 128)):
    t, f = shape
    if pattern == 'high_top':
        spec = np.concatenate([np.random.randn(t, f//2)*0.1+0.3, np.random.randn(t, f//2)*0.2+1.5], axis=1)
    elif pattern == 'high_bottom':
        spec = np.concatenate([np.random.randn(t, f//2)*0.2+1.5, np.random.randn(t, f//2)*0.1+0.3], axis=1)
    else:
        spec = np.random.randn(t, f)*0.3+0.8
    return np.clip(spec, 0, 2)

test_specs = {p: create_spec(p) for p in ['high_top', 'high_bottom', 'mixed']}
print(f"Generated {len(test_specs)} test specs")

## 5. Visualization Functions

In [ ]:
def visualize_importance_scores(tracker, title="Importance Scores"):
    importance = tracker.get_importance()
    stats = tracker.get_stats()
    
    plt.figure(figsize=(12, 4))
    plt.plot(importance, linewidth=2, color='purple')
    plt.fill_between(range(len(importance)), importance, alpha=0.3, color='purple')
    plt.axhline(stats['mean'], color='red', linestyle='--', label=f"Mean: {stats['mean']:.3f}")
    plt.xlabel('Frequency Bin')
    plt.ylabel('Importance')
    plt.title(title, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Stats: mean={stats['mean']:.3f}, std={stats['std']:.3f}")

print("Function 1 defined!")

In [ ]:
def visualize_multi_signal_weights(masker, spec, title="Multi-Signal Weights"):
    band_energies = masker._compute_band_energies(spec)
    masker._update_energy_ema(band_energies)
    energy_w = masker._compute_masking_weights(masker.band_energy_ema)
    
    importance_w = masker._compute_importance_weights(masker.importance_tracker.get_importance()) \
        if masker.importance_tracker else np.ones(len(energy_w))/len(energy_w)
    
    random_w = 1.0 - masker.energy_weight - masker.importance_weight - masker.afri_weight
    
    plt.figure(figsize=(12, 5))
    x = np.arange(len(energy_w))
    plt.bar(x, masker.energy_weight*energy_w, label=f'Energy ({masker.energy_weight:.0%})', color='blue', alpha=0.8)
    plt.bar(x, masker.importance_weight*importance_w, bottom=masker.energy_weight*energy_w,
            label=f'Importance ({masker.importance_weight:.0%})', color='purple', alpha=0.8)
    plt.xlabel('Band')
    plt.ylabel('Weight')
    plt.title(title, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

print("Function 2 defined!")

In [ ]:
def visualize_before_after_with_all_signals(orig, masked, energy, importance, title="Masking Result"):
    mask = (np.abs(orig - masked) > 0.01).astype(float)
    
    fig = plt.figure(figsize=(16, 4))
    gs = GridSpec(1, 4, width_ratios=[1, 1, 0.4, 0.4])
    
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(orig.T, aspect='auto', origin='lower', cmap='viridis')
    ax1.set_title('Original', fontweight='bold')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Frequency')
    
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(masked.T, aspect='auto', origin='lower', cmap='viridis')
    ax2.imshow(mask.T, aspect='auto', origin='lower', cmap='Reds', alpha=0.3)
    ax2.set_title(f'Masked ({mask.sum()/mask.size:.1%})', fontweight='bold')
    ax2.set_xlabel('Time')
    
    ax3 = fig.add_subplot(gs[2])
    ax3.barh(range(len(energy)), energy, color='blue', alpha=0.7)
    ax3.set_ylabel('Band')
    ax3.set_xlabel('Energy')
    ax3.set_title('Energy', fontweight='bold', fontsize=10)
    ax3.invert_yaxis()
    ax3.grid(alpha=0.3, axis='x')
    
    ax4 = fig.add_subplot(gs[3])
    band_imp = [importance[i*16:(i+1)*16].mean() for i in range(len(energy))]
    ax4.barh(range(len(band_imp)), band_imp, color='purple', alpha=0.7)
    ax4.set_xlabel('Importance')
    ax4.set_title('Importance', fontweight='bold', fontsize=10)
    ax4.invert_yaxis()
    ax4.grid(alpha=0.3, axis='x')
    
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Function 3 defined!")

In [ ]:
def visualize_importance_evolution(history, title="Importance Evolution"):
    history = np.array(history)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))
    
    im = ax1.imshow(history.T, aspect='auto', origin='lower', cmap='viridis')
    ax1.set_xlabel('Update')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Heatmap', fontweight='bold')
    plt.colorbar(im, ax=ax1)
    
    for bin in [20, 60, 100]:
        ax2.plot(history[:, bin], linewidth=2, label=f'Bin {bin}', alpha=0.8)
    ax2.set_xlabel('Update')
    ax2.set_ylabel('Importance')
    ax2.set_title('Selected Bins', fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Function 4 defined!")

In [ ]:
def visualize_weight_configuration_comparison(spec, tracker, configs):
    n = len(configs)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, (e_w, i_w, label) in enumerate(configs):
        m = ActivationGuidedFrequencyMasker(
            n_bands=2, mask_prob=1.0, max_band_width=16, n_mels=128,
            energy_weight=e_w, importance_weight=i_w, importance_tracker=tracker
        )
        masked = m(spec)
        mask = (np.abs(spec - masked) > 0.01).astype(float)
        
        axes[i].imshow(masked.T, aspect='auto', origin='lower', cmap='viridis')
        axes[i].imshow(mask.T, aspect='auto', origin='lower', cmap='Reds', alpha=0.3)
        axes[i].set_title(f'{label}\nE={e_w:.1f} I={i_w:.1f}\n{mask.sum()/mask.size:.1%}', fontsize=10)
        axes[i].set_xlabel('Time')
        axes[i].set_ylabel('Freq')
    
    for i in range(n, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Weight Configurations', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Function 5 defined!")

## 6. Examples

In [ ]:
visualize_importance_scores(tracker)

In [ ]:
visualize_multi_signal_weights(masker, test_specs['mixed'])

In [ ]:
for name, spec in test_specs.items():
    masked = masker(spec)
    energy = masker._compute_band_energies(spec)
    importance = tracker.get_importance()
    visualize_before_after_with_all_signals(spec, masked, energy, importance, f"Pattern: {name}")

In [ ]:
# Track evolution
t2 = FrequencyImportanceTracker(n_mels=128, ema_alpha=0.9, device=device)
t2.register_hook(model)
history = []
for i in range(30):
    s = torch.randn(1, 1, 100, 128).to(device)
    s[:, :, :, 80:] *= (2.0 if i < 15 else 1.0)
    with torch.no_grad():
        _ = model(s)
    history.append(t2.get_importance().copy())
visualize_importance_evolution(history)

In [ ]:
configs = [(1.0, 0.0, "Energy"), (0.0, 1.0, "Importance"), (0.5, 0.5, "Balanced"),
           (0.2, 0.7, "Imp-Heavy"), (0.7, 0.2, "Eng-Heavy"), (0.3, 0.3, "Hybrid")]
visualize_weight_configuration_comparison(test_specs['mixed'], tracker, configs)

In [ ]:
# Statistics
stats = masker.get_importance_stats()
print("\nMasker Statistics:")
print(f"Weights: {stats['weights']}")
print(f"Importance: {stats['importance_scores']}")
print(f"Tracker: {stats['tracker_stats']}")

## 7. Analysis

**Key Points:**
- Uses model activations to identify important frequencies
- Masks high-importance regions (opposite of energy-based)
- Combines energy + importance + random signals
- EMA smoothing for stability

**Comparison:**
- Phase 1: Random masking
- Phase 2: Energy-based (masks low energy)
- Phase 3: Activation-guided (masks high importance)

**AFRI Integration:**
- Set `afri_weight > 0` and provide `afri_profiles`
- Adds class-specific frequency guidance
- See `visualize_afri_profiles.ipynb`

## 8. Custom Analysis

In [ ]:
# Add your experiments here